In [1]:
from dataset import Dataset
from feature_extractor import FeatureExtractor
from evaluator import Evaluator

from Monitors import (
    MSPMonitor,
    OTBMonitor,
    GaussianMixtureMonitor,
    MahalanobisMonitor,
)

import torch

In [2]:
batch_size = 100
TORCH_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [3]:
# Define the model to test and layer to extract the features from
model = "densenet"
layer = 98

# Define the ID dataset, amongts 
# - CIFAR10
# - CIFAR100
# - SVHN
id_dataset = "cifar10"

# Define the OOD dataset, amongts
# - For CIFAR10  -> {CIFAR100, SVHN, LSUN}
# - for CIFAR100 -> {CIFAR10, LSUN, SVHN}
# - for SVHN     -> {CIFAR10, LSUN, tinyImageNet}
ood_dataset = "cifar10"

# Define the perturbation to apply (covariance shift)
perturbation = None

# Define the adversarial attack
adver_attack = "fgsm"

In [4]:
# Load the datasets
dataset_train = Dataset(id_dataset, "train", model, batch_size=batch_size)
dataset_test = Dataset(id_dataset, "test", model, batch_size=batch_size)
dataset_ood = Dataset(ood_dataset, "test", model, perturbation, adver_attack, batch_size=batch_size)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Define the feature extractor
feature_extractor = FeatureExtractor(model, id_dataset, [layer], TORCH_DEVICE)

c:\Users\MathieuDARIO\Workspace\neural-network-monitoring-benchmark\feature_extractor.py:312: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.

In [6]:
# Extract the feature to train and test the monitor
features_train, logits_train, softmax_train, \
    preds_train, labels_train = feature_extractor.get_features(dataset_train)
features_test, logits_test, softmax_test, \
    preds_test, labels_test = feature_extractor.get_features(dataset_test)
features_ood, logits_ood, softmax_ood, \
    preds_ood, labels_ood = feature_extractor.get_features(dataset_ood)

In [7]:
# Define an evaluator of the monitor, according the either :
# - OOD paradigm
# - OMS paradigm
eval_oms = Evaluator("oms", is_novelty=(id_dataset != ood_dataset))
eval_oms.fit_ground_truth(labels_test, labels_ood, preds_test, preds_ood)

eval_ood = Evaluator("ood", is_novelty=(id_dataset != ood_dataset))
eval_ood.fit_ground_truth(labels_test, labels_ood, preds_test, preds_ood)

In [8]:
# Define the monitor to evaluate
monitor = MahalanobisMonitor(id_dataset, model, layer, is_tied=False)
monitor.fit(features_train[0], preds_train, labels_train, save=True)

In [9]:
# Get the prediction of the monitor (i.e. whether we trust the model or not)
scores_test = monitor.predict(features_test[0], preds_test)
scores_ood = monitor.predict(features_ood[0], preds_ood)

In [10]:
# Get the metrics on the monitor's performance
p_oms, r_oms, f1_oms = eval_oms.get_metrics_f1opt(scores_test, scores_ood)
p_ood, r_ood, f1_ood = eval_ood.get_metrics_f1opt(scores_test, scores_ood)

aupr_oms = eval_oms.get_metric_aupr(scores_test, scores_ood)
aupr_ood = eval_ood.get_metric_aupr(scores_test, scores_ood)

auroc_oms = eval_oms.get_metric_auroc(scores_test, scores_ood)
auroc_ood = eval_ood.get_metric_auroc(scores_test, scores_ood)

tnr95tpr_oms = eval_oms.get_metric_tnr_frac_tpr(scores_test, scores_ood, frac=0.95)
tnr95tpr_ood = eval_ood.get_metric_tnr_frac_tpr(scores_test, scores_ood, frac=0.95)

In [12]:
print(f"""
Evaluation metrics for the {monitor.__class__} monitor:
---
OOD: 
    precision={p_ood:.4f}, recall={r_ood:.4f}, f1-score={f1_ood:.4f}
    aupr={aupr_ood},
    auroc={auroc_ood},
    tnr95tpr={tnr95tpr_ood},

OMS: 
    precision={p_oms:.4f}, recall={r_oms:.4f}, f1-score={f1_oms:.4f}
    aupr={aupr_oms},
    auroc={auroc_oms},
    tnr95tpr={tnr95tpr_oms},
---
""")


Evaluation metrics for the <class 'Monitors.mahalanobis.MahalanobisMonitor'> monitor:
---
OOD: 
    precision=0.7878, recall=0.8444, f1-score=0.8151
    aupr=0.32256360392327693,
    auroc=0.11714096,
    tnr95tpr=0.0004,

OMS: 
    precision=0.5173, recall=0.8258, f1-score=0.6362
    aupr=0.1686131352177318,
    auroc=0.17480000339603408,
    tnr95tpr=0.011733848702374372,
---

